# 3분 브리핑 제목 만들기

`news_macro_hot_issue`가 만든 시장 핵심 3줄로, 화면(피그마 홈 '오늘의 핵심 3가지', 3분 브리핑 창 199:121)에 필요한
**제목 한 줄**, **줄마다 소제목**, **키워드 3개**를 만든다. 3줄에 없는 사실은 쓰지 않는다.
결과는 `source/live/macro_headlines.jsonl`에 날짜마다 한 줄로 쌓이고, `news_to_db`가 `news.hot_macro`에 붙인다.
서버에서는 `run_news.py summaries`가 시장 요약 바로 뒤에 부른다(OpenAI 1회, 1센트 미만).

In [ ]:
import datetime as dt, json, os, urllib.error, urllib.request
from pathlib import Path

ROOT = Path.cwd()
PROJ = next((p for p in ROOT.parents if (p / '.env').exists()), ROOT.parents[0])
LIVE = ROOT / 'source' / 'live'
HOT_PATH = LIVE / 'macro_hot_issues.jsonl'
OUT_PATH = LIVE / 'macro_headlines.jsonl'
LLM_MODEL = 'gpt-6.1-terra'
RUN_LLM = False          # True여야 OpenAI를 부른다
KST = dt.timezone(dt.timedelta(hours=9))


def env_value(name):
    if os.environ.get(name):
        return os.environ[name]
    for line in (PROJ / '.env').read_text(encoding='utf-8').splitlines():
        if line.startswith(name + '='):
            return line.split('=', 1)[1].strip().strip('"').strip("'")
    return None

## 지시문과 호출

In [ ]:
PROMPT = (ROOT / 'prompt' / 'headline_prompt.md').read_text(encoding='utf-8')   # 지시문 파일(2026-11-20 노트북에서 옮김)


def ask(lines):
    body = {'model': LLM_MODEL, 'instructions': PROMPT, 'input': 'json 입력 문장:\n' + '\n'.join(f'{i + 1}. {l}' for i, l in enumerate(lines)),
            'text': {'format': {'type': 'json_object'}}, 'store': False}
    req = urllib.request.Request('https://api.openai.com/v1/responses', data=json.dumps(body).encode(), method='POST',
                                 headers={'Authorization': 'Bearer ' + (env_value('OPENAI_API_KEY') or ''), 'Content-Type': 'application/json'})
    try:
        with urllib.request.urlopen(req, timeout=120) as r:
            res = json.loads(r.read())
    except urllib.error.HTTPError as e:
        raise RuntimeError(f'OpenAI {e.code}: {e.read()[:300]!r}') from None
    text = ''.join(x.get('text', '') for item in res.get('output', []) if item.get('type') == 'message' for x in item.get('content', []))
    return json.loads(text), res.get('usage') or {}

## 실행: 가장 최근 시장 핵심 3줄마다(이미 만든 것은 건너뜀)

In [ ]:
def make_headlines():
    hot = [json.loads(l) for l in HOT_PATH.read_text(encoding='utf-8').splitlines() if l.strip()] if HOT_PATH.exists() else []
    latest = {}
    for h in hot:
        if h.get('lines') and str(h.get('created_kst', '')) >= str(latest.get(h['date'], {}).get('created_kst', '')):
            latest[h['date']] = h
    done = {(d['date'], d['source_created_kst']) for d in map(json.loads, OUT_PATH.read_text(encoding='utf-8').splitlines())} if OUT_PATH.exists() else set()
    made = 0
    for date, h in sorted(latest.items()):
        if (date, h.get('created_kst')) in done or not RUN_LLM:
            continue
        out, usage = ask(h['lines'])
        row = dict(date=date, created_kst=dt.datetime.now(KST).strftime('%Y-%m-%d %H:%M:%S'), source_created_kst=h.get('created_kst'),
                   title=str(out.get('title', '')).strip(), headings=[str(x) for x in out.get('headings', [])][:3],
                   keywords=[str(x) for x in out.get('keywords', [])][:3], model=LLM_MODEL, usage=usage)
        with OUT_PATH.open('a', encoding='utf-8') as f:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
        made += 1
        print(date, row['title'], row['headings'])
    print('제목 만든 날짜', made)


make_headlines()